In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm


print("All imports successful")

All imports successful


In [3]:
!pip install statsmodels

In [4]:
mh = pd.read_csv('13100465.csv')       # mental health table
unemp = pd.read_csv('14100017.csv')    # unemployment table

# first look at structure
print(mh.shape)
print(mh.columns.tolist())
mh.head()



(160992, 18)
['REF_DATE', 'GEO', 'DGUID', 'Age group', 'Gender', 'Indicators', 'Characteristics', 'UOM', 'UOM_ID', 'SCALAR_FACTOR', 'SCALAR_ID', 'VECTOR', 'COORDINATE', 'VALUE', 'STATUS', 'SYMBOL', 'TERMINATED', 'DECIMALS']


,REF_DATE,GEO,DGUID,Age group,Gender,Indicators,Characteristics,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Major depressive episode, life",Number of persons,Number,223,units,0,v73953257,1.1.1.5.1,3037049.0,NaN,NaN,NaN,0
1,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Major depressive episode, life","Low 95% confidence interval, number of persons",Number,223,units,0,v73953258,1.1.1.5.2,2913405.0,NaN,NaN,NaN,0
2,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Major depressive episode, life","High 95% confidence interval, number of persons",Number,223,units,0,v73953259,1.1.1.5.3,3160692.0,NaN,NaN,NaN,0
3,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Major depressive episode, life",Coefficient of variation for number of persons,Number,223,units,0,v73953260,1.1.1.5.4,21.0,NaN,NaN,NaN,0
4,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Major depressive episode, life",Percent,Percent,239,units,0,v73953261,1.1.1.5.5,12.2,NaN,NaN,NaN,1


In [5]:
mh_clean = mh[~mh['GEO'].isin(['Atlantic Provinces', 'Prairie Provinces'])]
print(mh_clean['GEO'].nunique())  # should be 11 (Canada + 10 provinces)

11


In [6]:
mh_clean['GEO'].unique()


<StringArray>
[                   'Canada', 'Newfoundland and Labrador',
      'Prince Edward Island',               'Nova Scotia',
             'New Brunswick',                    'Quebec',
                   'Ontario',                  'Manitoba',
              'Saskatchewan',                   'Alberta',
          'British Columbia']
Length: 11, dtype: str

In [7]:
print(unemp.shape)
print(unemp.columns.tolist())
unemp.head()

unemp["REF_DATE"].unique()

(4414080, 17)
['REF_DATE', 'GEO', 'DGUID', 'Labour force characteristics', 'Gender', 'Age group', 'UOM', 'UOM_ID', 'SCALAR_FACTOR', 'SCALAR_ID', 'VECTOR', 'COORDINATE', 'VALUE', 'STATUS', 'SYMBOL', 'TERMINATED', 'DECIMALS']


<StringArray>
['1976-01', '1976-02', '1976-03', '1976-04', '1976-05', '1976-06', '1976-07',
 '1976-08', '1976-09', '1976-10',
 ...
 '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05',
 '2026-06', '2026-07', '2026-08']
Length: 608, dtype: str

In [8]:
mh_clean. head()


,REF_DATE,GEO,DGUID,Age group,Gender,Indicators,Characteristics,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Major depressive episode, life",Number of persons,Number,223,units,0,v73953257,1.1.1.5.1,3037049.0,NaN,NaN,NaN,0
1,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Major depressive episode, life","Low 95% confidence interval, number of persons",Number,223,units,0,v73953258,1.1.1.5.2,2913405.0,NaN,NaN,NaN,0
2,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Major depressive episode, life","High 95% confidence interval, number of persons",Number,223,units,0,v73953259,1.1.1.5.3,3160692.0,NaN,NaN,NaN,0
3,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Major depressive episode, life",Coefficient of variation for number of persons,Number,223,units,0,v73953260,1.1.1.5.4,21.0,NaN,NaN,NaN,0
4,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Major depressive episode, life",Percent,Percent,239,units,0,v73953261,1.1.1.5.5,12.2,NaN,NaN,NaN,1


In [9]:
mh_pct = mh_clean[mh_clean['Characteristics'] == 'Percent'].copy()
unemp_rate = unemp[unemp['Labour force characteristics'] == 'Unemployment rate'].copy()

In [10]:
print(mh_pct.shape)
print(unemp_rate.shape)


(17919, 18)
(441408, 17)


In [11]:
unemp_rate['Gender'] = unemp_rate['Gender'].replace({'Total – Gender': 'Total, gender of person'})
unemp_rate['Age group'] = unemp_rate['Age group'].replace({'15 years and over': 'Total, 15 years and over'})

In [12]:
print(unemp_rate['Gender'].unique())
print(unemp_rate['Age group'].unique())

<StringArray>
['Total - Gender', 'Men+', 'Women+']
Length: 3, dtype: str
<StringArray>
['Total, 15 years and over',           '15 to 24 years',
        '25 years and over',           '25 to 44 years',
           '45 to 64 years',        '45 years and over',
           '25 to 54 years',        '55 years and over',
           '15 to 64 years',           '15 to 19 years',
           '20 to 24 years',           '25 to 29 years',
           '30 to 34 years',           '35 to 39 years',
           '40 to 44 years',           '45 to 49 years',
           '50 to 54 years',           '55 to 59 years',
           '60 to 64 years',        '65 years and over',
           '65 to 69 years',        '70 years and over']
Length: 22, dtype: str


In [13]:
unemp_rate['Gender'] = unemp_rate['Gender'].replace({'Total - Gender': 'Total, gender of person'})

print(unemp_rate['Gender'].unique())

<StringArray>
['Total, gender of person', 'Men+', 'Women+']
Length: 3, dtype: str


In [14]:
unemp_rate['REF_DATE'] = pd.to_datetime(unemp_rate['REF_DATE'])
unemp_rate['Year'] = unemp_rate['REF_DATE'].dt.year

unemp_annual = (
    unemp_rate[unemp_rate['Year'].isin([2002, 2012, 2022])]
    .groupby(['GEO', 'Age group', 'Gender', 'Year'])['VALUE']
    .mean()
    .reset_index()
)

print(unemp_annual.shape)
unemp_annual.head()

(2178, 5)


,GEO,Age group,Gender,Year,VALUE
0,Alberta,15 to 19 years,Men+,2002,17.791667
1,Alberta,15 to 19 years,Men+,2012,14.516667
2,Alberta,15 to 19 years,Men+,2022,14.475000
3,Alberta,15 to 19 years,"Total, gender of person",2002,15.391667
4,Alberta,15 to 19 years,"Total, gender of person",2012,12.991667


In [15]:
#check reliability flags on mental health data: 

print(mh_pct['STATUS'].unique())

<StringArray>
[nan, 'E', 'F', 'x']
Length: 4, dtype: str


In [16]:
mh_reliable = mh_pct[~mh_pct['STATUS'].isin(['F', 'x'])].copy()
print(mh_reliable.shape)

(10676, 18)


In [17]:

mh_reliable = mh_reliable.drop(columns=['SYMBOL'])


In [18]:
print(mh_pct.isna().sum())


REF_DATE               0
GEO                    0
DGUID                  0
Age group              0
Gender                 0
Indicators             0
Characteristics        0
UOM                    0
UOM_ID                 0
SCALAR_FACTOR          0
SCALAR_ID              0
VECTOR                 0
COORDINATE             0
VALUE               7243
STATUS              7661
SYMBOL             17919
TERMINATED         17919
DECIMALS               0
dtype: int64


In [19]:

print(unemp_annual.isna().sum())

GEO           0
Age group     0
Gender        0
Year          0
VALUE        99
dtype: int64


In [20]:
print(mh_pct[mh_pct['VALUE'].isna()]['STATUS'].value_counts(dropna=False))

STATUS
x    4103
F    3140
Name: count, dtype: int64


In [21]:
print(mh_reliable['Indicators'].value_counts())

Indicators
Perceived mental health, very good or excellent                                     493
Perceived mental health, good                                                       487
Cannabis use, life                                                                  475
Self-rated work stress, days not at all or not very stressful                       458
Suicidal thoughts, life                                                             450
Self-rated work stress, days a bit stressful                                        447
Perceived mental health, fair or poor                                               445
Major depressive episode, life                                                      427
Self-rated work stress, days quite a bit or extremely stressful                     412
Cannabis use, 12 months                                                             404
Major depressive episode, 12 months                                                 340
Suicidal thoughts, 12

In [22]:
mh_reliable = mh_reliable[~mh_reliable['STATUS'].isin(['F', 'X'])].copy()


In [23]:
print(mh_reliable.shape)
print(mh_reliable['VALUE'].isna().sum())

(10676, 17)
0


In [24]:
unemp_annual_clean = unemp_annual.dropna(subset=['VALUE'])
print (unemp_annual_clean.shape)

(2079, 5)


In [25]:
mh_pct.head(10)

,REF_DATE,GEO,DGUID,Age group,Gender,Indicators,Characteristics,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
4,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Major depressive episode, life",Percent,Percent,239,units,0,v73953261,1.1.1.5.5,12.2,NaN,NaN,NaN,1
12,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Major depressive episode, 12 months",Percent,Percent,239,units,0,v73953270,1.1.1.6.5,4.8,NaN,NaN,NaN,1
20,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Cannabis use, life",Percent,Percent,239,units,0,v73953279,1.1.1.19.5,41.3,NaN,NaN,NaN,1
28,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Cannabis use, 12 months",Percent,Percent,239,units,0,v73953288,1.1.1.20.5,12.2,NaN,NaN,NaN,1
36,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Suicidal thoughts, life",Percent,Percent,239,units,0,v73953297,1.1.1.23.5,13.4,NaN,NaN,NaN,1
44,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Suicidal thoughts, 12 months",Percent,Percent,239,units,0,v73953306,1.1.1.24.5,3.7,NaN,NaN,NaN,1
52,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Perceived mental health, very good or excellent",Percent,Percent,239,units,0,v73953315,1.1.1.25.5,67.1,NaN,NaN,NaN,1
60,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Perceived mental health, good",Percent,Percent,239,units,0,v73953324,1.1.1.26.5,26.1,NaN,NaN,NaN,1
68,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Perceived mental health, fair or poor",Percent,Percent,239,units,0,v73953333,1.1.1.27.5,6.9,NaN,NaN,NaN,1
76,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Self-rated work stress, days quite a bit or ex...",Percent,Percent,239,units,0,v73953342,1.1.1.28.5,30.7,NaN,NaN,NaN,1


In [26]:
mh_pct[['GEO', 'Age group', 'Gender', 'Indicators', 'VALUE']].head(10)

,GEO,Age group,Gender,Indicators,VALUE
4,Canada,"Total, 15 years and over","Total, gender of person","Major depressive episode, life",12.2
12,Canada,"Total, 15 years and over","Total, gender of person","Major depressive episode, 12 months",4.8
20,Canada,"Total, 15 years and over","Total, gender of person","Cannabis use, life",41.3
28,Canada,"Total, 15 years and over","Total, gender of person","Cannabis use, 12 months",12.2
36,Canada,"Total, 15 years and over","Total, gender of person","Suicidal thoughts, life",13.4
44,Canada,"Total, 15 years and over","Total, gender of person","Suicidal thoughts, 12 months",3.7
52,Canada,"Total, 15 years and over","Total, gender of person","Perceived mental health, very good or excellent",67.1
60,Canada,"Total, 15 years and over","Total, gender of person","Perceived mental health, good",26.1
68,Canada,"Total, 15 years and over","Total, gender of person","Perceived mental health, fair or poor",6.9
76,Canada,"Total, 15 years and over","Total, gender of person","Self-rated work stress, days quite a bit or ex...",30.7


In [27]:
print(mh_pct['Characteristics'].unique())


<StringArray>
['Percent']
Length: 1, dtype: str


In [28]:
mh_final = mh_reliable[['GEO', 'REF_DATE', 'Age group', 'Gender', 'Indicators', 'VALUE']].copy()
mh_final = mh_final.rename(columns={'REF_DATE': 'Year', 'VALUE': 'Mental Health Percent'})

print(mh_final.shape)
mh_final.head(10)

(10676, 6)


,GEO,Year,Age group,Gender,Indicators,Mental Health Percent
4,Canada,2002,"Total, 15 years and over","Total, gender of person","Major depressive episode, life",12.2
12,Canada,2002,"Total, 15 years and over","Total, gender of person","Major depressive episode, 12 months",4.8
20,Canada,2002,"Total, 15 years and over","Total, gender of person","Cannabis use, life",41.3
28,Canada,2002,"Total, 15 years and over","Total, gender of person","Cannabis use, 12 months",12.2
36,Canada,2002,"Total, 15 years and over","Total, gender of person","Suicidal thoughts, life",13.4
44,Canada,2002,"Total, 15 years and over","Total, gender of person","Suicidal thoughts, 12 months",3.7
52,Canada,2002,"Total, 15 years and over","Total, gender of person","Perceived mental health, very good or excellent",67.1
60,Canada,2002,"Total, 15 years and over","Total, gender of person","Perceived mental health, good",26.1
68,Canada,2002,"Total, 15 years and over","Total, gender of person","Perceived mental health, fair or poor",6.9
76,Canada,2002,"Total, 15 years and over","Total, gender of person","Self-rated work stress, days quite a bit or ex...",30.7


In [29]:
unemp_final = unemp_annual_clean.rename(columns={'VALUE': 'Unemployment Rate'})
print(unemp_final.shape)
unemp_final.head(10)

(2079, 5)


,GEO,Age group,Gender,Year,Unemployment Rate
0,Alberta,15 to 19 years,Men+,2002,17.791667
1,Alberta,15 to 19 years,Men+,2012,14.516667
2,Alberta,15 to 19 years,Men+,2022,14.475000
3,Alberta,15 to 19 years,"Total, gender of person",2002,15.391667
4,Alberta,15 to 19 years,"Total, gender of person",2012,12.991667
5,Alberta,15 to 19 years,"Total, gender of person",2022,14.116667
6,Alberta,15 to 19 years,Women+,2002,12.758333
7,Alberta,15 to 19 years,Women+,2012,11.341667
8,Alberta,15 to 19 years,Women+,2022,13.675000
9,Alberta,15 to 24 years,Men+,2002,12.108333


In [30]:
merged = pd.merge(
    mh_final,
    unemp_final,
    on=['GEO', 'Age group', 'Gender', 'Year'],
    how='inner'
)

print(merged.shape)
merged.head(10)

(9093, 7)


,GEO,Year,Age group,Gender,Indicators,Mental Health Percent,Unemployment Rate
0,Canada,2002,"Total, 15 years and over","Total, gender of person","Major depressive episode, life",12.2,7.675
1,Canada,2002,"Total, 15 years and over","Total, gender of person","Major depressive episode, 12 months",4.8,7.675
2,Canada,2002,"Total, 15 years and over","Total, gender of person","Cannabis use, life",41.3,7.675
3,Canada,2002,"Total, 15 years and over","Total, gender of person","Cannabis use, 12 months",12.2,7.675
4,Canada,2002,"Total, 15 years and over","Total, gender of person","Suicidal thoughts, life",13.4,7.675
5,Canada,2002,"Total, 15 years and over","Total, gender of person","Suicidal thoughts, 12 months",3.7,7.675
6,Canada,2002,"Total, 15 years and over","Total, gender of person","Perceived mental health, very good or excellent",67.1,7.675
7,Canada,2002,"Total, 15 years and over","Total, gender of person","Perceived mental health, good",26.1,7.675
8,Canada,2002,"Total, 15 years and over","Total, gender of person","Perceived mental health, fair or poor",6.9,7.675
9,Canada,2002,"Total, 15 years and over","Total, gender of person","Self-rated work stress, days quite a bit or ex...",30.7,7.675


In [31]:
merged.head()

,GEO,Year,Age group,Gender,Indicators,Mental Health Percent,Unemployment Rate
0,Canada,2002,"Total, 15 years and over","Total, gender of person","Major depressive episode, life",12.2,7.675
1,Canada,2002,"Total, 15 years and over","Total, gender of person","Major depressive episode, 12 months",4.8,7.675
2,Canada,2002,"Total, 15 years and over","Total, gender of person","Cannabis use, life",41.3,7.675
3,Canada,2002,"Total, 15 years and over","Total, gender of person","Cannabis use, 12 months",12.2,7.675
4,Canada,2002,"Total, 15 years and over","Total, gender of person","Suicidal thoughts, life",13.4,7.675


In [32]:
correlation = merged['Unemployment Rate'].corr(merged['Mental Health Percent'])
print(f"Overall correlation: {correlation:.3f}")

Overall correlation: 0.071


mh_finals["Indicator"].unique()

In [33]:
from scipy import stats

results = []
for indicator in merged['Indicators'].unique(): # Iterate through each unique indicator
    subset = merged[merged['Indicators'] == indicator] # Select rows for the current indicator
    if len(subset) > 5:     # Only compute correlation if there are more than 5 data points
        r, p = stats.pearsonr(subset['Unemployment Rate'], subset['Mental Health Percent'])
        results.append({'Indicator': indicator, 'Correlation': r, 'P-value': p, 'N': len(subset)})

results_df = pd.DataFrame(results).sort_values('Correlation', ascending=False)
print(results_df.to_string())

                                                                           Indicator  Correlation       P-value    N
29                    Other drug abuse or dependence (excluding cannabis), 12 months     0.843337  4.906460e-09   30
27                                           Cannabis abuse or dependence, 12 months     0.816353  3.543006e-16   63
23                           Any substance use disorder (alcohol or drug), 12 months     0.676589  4.527206e-20  140
44                                                          Social phobia, 12 months     0.662244  2.019239e-08   57
40  Any selected disorder (mental or substance), including social phobia,  12 months     0.622670  8.960920e-10   79
25                                            Alcohol abuse or dependence, 12 months     0.622361  6.470779e-13  108
42                                                   Any anxiety disorder, 12 months     0.549323  7.017686e-07   71
43                                                              

In [34]:
# unemployment side should be unique on the join keys (expect 0)
print(unemp_final.duplicated(subset=['GEO', 'Age group', 'Gender', 'Year']).sum())

# see exactly which rows didn't match
check = pd.merge(mh_final, unemp_final, on=['GEO', 'Age group', 'Gender', 'Year'],
                 how='left', indicator=True)
print(check['_merge'].value_counts())
print(check[check['_merge'] == 'left_only'][['GEO', 'Age group', 'Gender', 'Year']].drop_duplicates())

0
_merge
both          9093
left_only     1583
right_only       0
Name: count, dtype: int64
                            GEO       Age group                   Gender  Year
82                       Canada  25 to 64 years  Total, gender of person  2002
96                       Canada  25 to 64 years                     Men+  2002
110                      Canada  25 to 64 years                   Women+  2002
306   Newfoundland and Labrador  25 to 64 years  Total, gender of person  2002
318   Newfoundland and Labrador  25 to 64 years                     Men+  2002
...                         ...             ...                      ...   ...
6959                    Alberta  25 to 64 years                     Men+  2012
6992                    Alberta  25 to 64 years                   Women+  2012
7461           British Columbia  25 to 64 years  Total, gender of person  2012
7499           British Columbia  25 to 64 years                     Men+  2012
7536           British Columbia  25 to 

In [35]:
missing_65 = check[(check['_merge'] == 'left_only') & (check['Age group'] == '65 years and over')]
print(missing_65[['GEO', 'Gender', 'Year']].drop_duplicates())

# compare with what was dropped from the unemployment side
dropped = unemp_annual[unemp_annual['VALUE'].isna()]
print(dropped[dropped['Age group'] == '65 years and over'][['GEO', 'Gender', 'Year']])

                            GEO                   Gender  Year
406   Newfoundland and Labrador  Total, gender of person  2002
410   Newfoundland and Labrador                     Men+  2002
412   Newfoundland and Labrador                   Women+  2002
559        Prince Edward Island  Total, gender of person  2002
563        Prince Edward Island                     Men+  2002
565        Prince Edward Island                   Women+  2002
756                 Nova Scotia                   Women+  2002
933               New Brunswick  Total, gender of person  2002
938               New Brunswick                     Men+  2002
940               New Brunswick                   Women+  2002
1149                     Quebec                   Women+  2002
1574                   Manitoba                     Men+  2002
1755               Saskatchewan  Total, gender of person  2002
1763               Saskatchewan                     Men+  2002
1768               Saskatchewan                   Women

In [36]:
missing_65 = check[(check['_merge'] == 'left_only') & (check['Age group'] == '65 years and over')]
print(missing_65[['GEO', 'Gender', 'Year']].drop_duplicates())

# compare with what was dropped from the unemployment side
dropped = unemp_annual[unemp_annual['VALUE'].isna()]
print(dropped[dropped['Age group'] == '65 years and over'][['GEO', 'Gender', 'Year']])

                            GEO                   Gender  Year
406   Newfoundland and Labrador  Total, gender of person  2002
410   Newfoundland and Labrador                     Men+  2002
412   Newfoundland and Labrador                   Women+  2002
559        Prince Edward Island  Total, gender of person  2002
563        Prince Edward Island                     Men+  2002
565        Prince Edward Island                   Women+  2002
756                 Nova Scotia                   Women+  2002
933               New Brunswick  Total, gender of person  2002
938               New Brunswick                     Men+  2002
940               New Brunswick                   Women+  2002
1149                     Quebec                   Women+  2002
1574                   Manitoba                     Men+  2002
1755               Saskatchewan  Total, gender of person  2002
1763               Saskatchewan                     Men+  2002
1768               Saskatchewan                   Women

In [37]:
a = set(map(tuple, missing_65[['GEO', 'Gender', 'Year']].drop_duplicates().values))
b = set(map(tuple, dropped[dropped['Age group'] == '65 years and over'][['GEO', 'Gender', 'Year']].values))

print(len(a), len(b))
print(a - b)   # groups unmatched in the merge but NOT dropped for missing unemployment

17 17
set()


In [38]:
clean = merged[
    (merged['Gender'].isin(['Men+', 'Women+'])) &
    (merged['Age group'].isin(['15 to 24 years', '25 to 44 years', '45 to 64 years', '65 years and over'])) &
    (merged['GEO'] != 'Canada')
].copy()

print(clean.shape)
print(clean['GEO'].nunique(), clean['Age group'].nunique(), clean['Gender'].nunique(), clean['Year'].nunique())

(3299, 7)
10 4 2 3


In [39]:
results = []
for indicator in clean['Indicators'].unique():
    subset = clean[clean['Indicators'] == indicator]
    if len(subset) > 5:
        r, p = stats.pearsonr(subset['Unemployment Rate'], subset['Mental Health Percent'])
        results.append({'Indicator': indicator, 'Correlation': r, 'P-value': p, 'N': len(subset)})

results_df = pd.DataFrame(results).sort_values('Correlation', ascending=False)
print(results_df.to_string())

                                                                           Indicator  Correlation       P-value    N
29                                           Cannabis abuse or dependence, 12 months     0.772519  2.463665e-02    8
17                           Any substance use disorder (alcohol or drug), 12 months     0.662818  5.788608e-06   38
42                                                          Social phobia, 12 months     0.599550  3.934904e-02   12
38  Any selected disorder (mental or substance), including social phobia,  12 months     0.552544  2.801761e-03   27
34                                            Alcohol abuse or dependence, 12 months     0.525945  6.925969e-03   25
5                      Self-rated work stress, days not at all or not very stressful     0.493173  1.278834e-11  167
1                                                            Cannabis use, 12 months     0.490270  2.782805e-09  131
11                                                      Suicidal

In [40]:
print(merged['Gender'].unique())
print(merged.shape)

<StringArray>
['Total, gender of person', 'Men+', 'Women+']
Length: 3, dtype: str
(9093, 7)


In [41]:
groups = ['15 to 24 years', '25 to 44 years', '45 to 64 years', '65 years and over']
base = merged[(merged['Age group'].isin(groups)) & (merged['GEO'] != 'Canada')]

def run_corr(df, label):
    rows = []
    for ind in df['Indicators'].unique():
        sub = df[df['Indicators'] == ind]
        if len(sub) > 20:
            r, p = stats.pearsonr(sub['Unemployment Rate'], sub['Mental Health Percent'])
            rows.append({'Indicator': ind, f'r_{label}': r, f'p_{label}': p, f'N_{label}': len(sub)})
    return pd.DataFrame(rows)

total = run_corr(base[base['Gender'] == 'Total, gender of person'], 'total')
men   = run_corr(base[base['Gender'] == 'Men+'], 'men')
women = run_corr(base[base['Gender'] == 'Women+'], 'women')

compare = total.merge(men, on='Indicator').merge(women, on='Indicator')
compare['men_vs_women_gap'] = compare['r_men'] - compare['r_women']
print(compare.sort_values('men_vs_women_gap').to_string())

                                                                          Indicator   r_total       p_total  N_total     r_men         p_men  N_men   r_women       p_women  N_women  men_vs_women_gap
17                               Any substance use disorder (alcohol or drug), life -0.126230  3.202633e-01       64 -0.230653  9.995001e-02     52  0.065497  6.582792e-01       48         -0.296150
7                                                     Perceived mental health, good -0.479336  7.778363e-07       96 -0.534948  5.584866e-08     90 -0.291683  5.547236e-03       89         -0.243266
18                                                Alcohol abuse or dependence, life -0.221529  8.893006e-02       60 -0.242361  8.343060e-02     52 -0.008224  9.567414e-01       46         -0.234137
10                                     Self-rated work stress, days a bit stressful -0.209340  4.767999e-02       90 -0.363248  7.375218e-04     83 -0.141981  2.090063e-01       80         -0.221268
21   

In [42]:
import statsmodels.formula.api as smf   # !pip install statsmodels if needed

keep = ['Cannabis use, 12 months',
        'Suicidal thoughts, 12 months',
        'Major depressive episode, 12 months',
        'Any selected disorder (mental or substance), excluding social phobia, 12 months',
        'Other drug use (excluding cannabis), 12 months',
        'Perceived mental health, very good or excellent']

rows = []
for ind in keep:
    sub = clean[clean['Indicators'] == ind].rename(
        columns={'Unemployment Rate': 'unemp', 'Mental Health Percent': 'mh', 'Age group': 'age'})
    m = smf.ols('mh ~ unemp * C(Gender) + C(age) + C(Year)', data=sub).fit()
    rows.append({'Indicator': ind,
                 'slope_men': m.params['unemp'], 'p_men': m.pvalues['unemp'],
                 'women_minus_men': m.params['unemp:C(Gender)[T.Women+]'],
                 'p_gender_diff': m.pvalues['unemp:C(Gender)[T.Women+]'],
                 'N': len(sub)})

print(pd.DataFrame(rows).to_string())

                                                                         Indicator  slope_men     p_men  women_minus_men  p_gender_diff    N
0                                                          Cannabis use, 12 months   0.018640  0.898600         0.233570       0.279392  131
1                                                     Suicidal thoughts, 12 months  -0.205181  0.033416         0.257473       0.017908   63
2                                              Major depressive episode, 12 months  -0.227683  0.046613         0.063560       0.614616   87
3  Any selected disorder (mental or substance), excluding social phobia, 12 months   0.020771  0.900584        -0.413810       0.092905   81
4                                   Other drug use (excluding cannabis), 12 months   0.202314  0.171215        -0.261720       0.143993   69
5                                  Perceived mental health, very good or excellent   0.609909  0.000294        -0.206753       0.379235  181


In [43]:
rows = []
for ind in keep:
    sub = clean[clean['Indicators'] == ind].rename(
        columns={'Unemployment Rate': 'unemp', 'Mental Health Percent': 'mh', 'Age group': 'age'})
    m = smf.ols('mh ~ unemp * C(Gender) + C(age) + C(Year) + C(GEO)', data=sub).fit(
        cov_type='cluster', cov_kwds={'groups': sub['GEO']})
    rows.append({'Indicator': ind,
                 'slope_men': m.params['unemp'], 'p_men': m.pvalues['unemp'],
                 'women_minus_men': m.params['unemp:C(Gender)[T.Women+]'],
                 'p_gender_diff': m.pvalues['unemp:C(Gender)[T.Women+]'],
                 'N': len(sub)})

fe_results = pd.DataFrame(rows)
print(fe_results.to_string())

                                                                         Indicator  slope_men     p_men  women_minus_men  p_gender_diff    N
0                                                          Cannabis use, 12 months   0.124621  0.764608         0.311975       0.109259  131
1                                                     Suicidal thoughts, 12 months   0.104758  0.433539         0.402834       0.000152   63
2                                              Major depressive episode, 12 months  -0.144028  0.398724         0.068148       0.650964   87
3  Any selected disorder (mental or substance), excluding social phobia, 12 months   0.042810  0.859269        -0.310212       0.036331   81
4                                   Other drug use (excluding cannabis), 12 months   0.382589  0.257093        -0.190118       0.440947   69
5                                  Perceived mental health, very good or excellent   0.514972  0.012286        -0.303158       0.069890  181


In [44]:
spec = {
    'A: no controls': 'mh ~ unemp',
    'B: + gender, age, year': 'mh ~ unemp + C(Gender) + C(age) + C(Year)',
    'C: + province (clustered)': 'mh ~ unemp + C(Gender) + C(age) + C(Year) + C(GEO)',
}

rows = []
for ind in keep:
    sub = clean[clean['Indicators'] == ind].rename(
        columns={'Unemployment Rate': 'unemp', 'Mental Health Percent': 'mh', 'Age group': 'age'})
    for name, f in spec.items():
        if name.startswith('C'):
            m = smf.ols(f, data=sub).fit(cov_type='cluster', cov_kwds={'groups': sub['GEO']})
        else:
            m = smf.ols(f, data=sub).fit()
        rows.append({'Indicator': ind, 'Model': name,
                     'Slope': m.params['unemp'], 'P': m.pvalues['unemp'], 'N': len(sub)})

robust = pd.DataFrame(rows)
print(robust.pivot(index='Indicator', columns='Model', values='Slope').round(3))
print(robust.pivot(index='Indicator', columns='Model', values='P').round(3))
robust.to_csv('robustness_table.csv', index=False)

Model                                               A: no controls  \
Indicator                                                            
Any selected disorder (mental or substance), ex...           0.565   
Cannabis use, 12 months                                      1.133   
Major depressive episode, 12 months                          0.155   
Other drug use (excluding cannabis), 12 months               0.309   
Perceived mental health, very good or excellent              0.765   
Suicidal thoughts, 12 months                                 0.235   

Model                                               B: + gender, age, year  \
Indicator                                                                    
Any selected disorder (mental or substance), ex...                  -0.029   
Cannabis use, 12 months                                              0.052   
Major depressive episode, 12 months                                 -0.202   
Other drug use (excluding cannabis), 12 months   

In [45]:
import json

to_save = {
    'merged_data.csv': 'merged',
    'unemp_final.csv': 'unemp_final',
    'clean_merged.csv': 'clean',
    'correlation_results.csv': 'results_df',
    'gender_comparison.csv': 'compare',
    'robustness_table.csv': 'robust',
}

for fname, var in to_save.items():
    if var in globals():
        globals()[var].to_csv(fname, index=False)
        print('saved  ', fname, globals()[var].shape)
    else:
        print('MISSING', var, '(not in memory)')

if 'keep' in globals():
    with open('keep_indicators.json', 'w') as f:
        json.dump(keep, f)
    print('saved   keep_indicators.json')

saved   merged_data.csv (9093, 7)
saved   unemp_final.csv (2079, 5)
saved   clean_merged.csv (3299, 7)
saved   correlation_results.csv (43, 4)
saved   gender_comparison.csv (24, 11)
saved   robustness_table.csv (18, 5)
saved   keep_indicators.json


In [46]:
levels = clean[clean['Indicators'].isin(keep)]

by_age = (levels.groupby(['Indicators', 'Age group'])['Mental Health Percent']
          .mean().unstack('Age group').round(1))
print(by_age)

Age group                                           15 to 24 years  \
Indicators                                                           
Any selected disorder (mental or substance), ex...            21.0   
Cannabis use, 12 months                                       30.5   
Major depressive episode, 12 months                            9.0   
Other drug use (excluding cannabis), 12 months                11.0   
Perceived mental health, very good or excellent               63.6   
Suicidal thoughts, 12 months                                   6.8   

Age group                                           25 to 44 years  \
Indicators                                                           
Any selected disorder (mental or substance), ex...            14.2   
Cannabis use, 12 months                                       17.6   
Major depressive episode, 12 months                            6.3   
Other drug use (excluding cannabis), 12 months                 8.1   
Perceived mental he

In [47]:
levels = clean[clean['Indicators'].isin(keep)]

by_gender = (levels.groupby(['Indicators', 'Gender'])['Mental Health Percent']
             .mean().unstack('Gender').round(1))

print(by_gender)

Gender                                              Men+  Women+
Indicators                                                      
Any selected disorder (mental or substance), ex...  14.0    14.5
Cannabis use, 12 months                             22.4    16.8
Major depressive episode, 12 months                  4.7     7.5
Other drug use (excluding cannabis), 12 months       8.4     6.8
Perceived mental health, very good or excellent     65.0    60.1
Suicidal thoughts, 12 months                         4.3     5.1


In [48]:
groups = ['15 to 24 years', '25 to 44 years', '45 to 64 years', '65 years and over']
levels_prov = merged[(merged['Age group'].isin(groups)) & (merged['GEO'] != 'Canada') &
                      (merged['Indicators'].isin(keep))]

total_by_prov = (levels_prov[levels_prov['Gender'] == 'Total, gender of person']
                  .groupby(['Indicators', 'GEO'])['Mental Health Percent']
                  .mean().unstack('GEO').round(1))
print(total_by_prov)

GEO                                                 Alberta  British Columbia  \
Indicators                                                                      
Any selected disorder (mental or substance), ex...     15.3              15.1   
Cannabis use, 12 months                                18.6              21.2   
Major depressive episode, 12 months                     7.1               5.8   
Other drug use (excluding cannabis), 12 months          7.6               8.2   
Perceived mental health, very good or excellent        59.4              57.9   
Suicidal thoughts, 12 months                            5.1               5.0   

GEO                                                 Manitoba  New Brunswick  \
Indicators                                                                    
Any selected disorder (mental or substance), ex...      17.8           11.9   
Cannabis use, 12 months                                 19.3           17.0   
Major depressive episode, 12 months

In [49]:
by_age_unemp = (unemp_final[unemp_final['Age group'].isin(
    ['15 to 24 years', '25 to 44 years', '45 to 64 years', '65 years and over'])]
    .groupby('Age group')['Unemployment Rate'].mean().round(1))

print(by_age_unemp)

Age group
15 to 24 years       13.1
25 to 44 years        6.7
45 to 64 years        6.5
65 years and over     7.6
Name: Unemployment Rate, dtype: float64


In [50]:
rows = []
for ind in keep:
    sub = clean[clean['Indicators'] == ind].rename(
        columns={'Unemployment Rate': 'unemp', 'Mental Health Percent': 'mh', 'Age group': 'age'})
    m = smf.ols('mh ~ unemp * C(GEO) + C(age) + C(Gender) + C(Year)', data=sub).fit()
    print(f"\n{ind}")
    print(m.summary().tables[1])


Cannabis use, 12 months
                                                coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------
Intercept                                    27.9426      4.235      6.597      0.000      19.545      36.341
C(GEO)[T.British Columbia]                    7.0399      3.589      1.961      0.052      -0.077      14.157
C(GEO)[T.Manitoba]                           -3.2346      4.050     -0.799      0.426     -11.266       4.797
C(GEO)[T.New Brunswick]                      -3.5190      4.613     -0.763      0.447     -12.666       5.628
C(GEO)[T.Newfoundland and Labrador]           9.1540      5.797      1.579      0.117      -2.340      20.648
C(GEO)[T.Nova Scotia]                         0.9125      4.099      0.223      0.824      -7.215       9.040
C(GEO)[T.Ontario]                             2.2325      3.506      0.637      0.526      -4.7

In [51]:
def province_slopes(ind):
    sub = clean[clean['Indicators'] == ind].rename(
        columns={'Unemployment Rate': 'unemp', 'Mental Health Percent': 'mh', 'Age group': 'age'})
    m = smf.ols('mh ~ unemp * C(GEO) + C(age) + C(Gender) + C(Year)', data=sub).fit()

    base_slope = m.params['unemp']  # this is Alberta's slope (the reference province)
    slopes = {}
    for p in sub['GEO'].unique():
        term = f'unemp:C(GEO)[T.{p}]'
        slopes[p] = base_slope + m.params[term] if term in m.params else base_slope
    return pd.Series(slopes, name=ind)

all_slopes = pd.DataFrame({ind: province_slopes(ind) for ind in keep})
print(all_slopes.round(2))

                           Cannabis use, 12 months  \
Alberta                                       0.42   
British Columbia                             -0.04   
Manitoba                                      0.66   
New Brunswick                                 0.58   
Newfoundland and Labrador                    -0.46   
Nova Scotia                                   0.54   
Ontario                                       0.02   
Prince Edward Island                         -0.94   
Quebec                                        1.27   
Saskatchewan                                  0.50   

                           Suicidal thoughts, 12 months  \
Alberta                                           -0.02   
British Columbia                                  -0.15   
Manitoba                                           0.04   
New Brunswick                                     -2.24   
Newfoundland and Labrador                           NaN   
Nova Scotia                                        

In [52]:
avg_rank = all_slopes.rank(ascending=False).mean(axis=1).sort_values()
print(avg_rank.round(1))

Saskatchewan                 3.2
Alberta                      3.5
Quebec                       3.7
Nova Scotia                  3.8
New Brunswick                4.7
Manitoba                     5.3
Ontario                      6.2
British Columbia             7.2
Newfoundland and Labrador    8.2
Prince Edward Island         8.3
dtype: float64


In [53]:
# z-score each indicator across provinces (row-wise)
z_by_prov = total_by_prov.sub(total_by_prov.mean(axis=1), axis=0).div(total_by_prov.std(axis=1), axis=0)

print(z_by_prov.round(2))

GEO                                                 Alberta  British Columbia  \
Indicators                                                                      
Any selected disorder (mental or substance), ex...     0.60              0.53   
Cannabis use, 12 months                                0.33              1.79   
Major depressive episode, 12 months                    1.06              0.13   
Other drug use (excluding cannabis), 12 months         0.54              0.90   
Perceived mental health, very good or excellent       -0.64             -0.95   
Suicidal thoughts, 12 months                           0.80              0.66   

GEO                                                 Manitoba  New Brunswick  \
Indicators                                                                    
Any selected disorder (mental or substance), ex...      1.50          -0.61   
Cannabis use, 12 months                                 0.73          -0.56   
Major depressive episode, 12 months

In [54]:
signature = {}
for prov in z_by_prov.columns:
    col = z_by_prov[prov]
    top_indicator = col.abs().idxmax()
    signature[prov] = {'Indicator': top_indicator, 'z-score': round(col[top_indicator], 2),
                        'Actual rate': total_by_prov.loc[top_indicator, prov]}

signature_df = pd.DataFrame(signature).T
print(signature_df)

                                                                   Indicator  \
Alberta                                  Major depressive episode, 12 months   
British Columbia                                     Cannabis use, 12 months   
Manitoba                                 Major depressive episode, 12 months   
New Brunswick                                   Suicidal thoughts, 12 months   
Newfoundland and Labrador    Perceived mental health, very good or excellent   
Nova Scotia                Any selected disorder (mental or substance), e...   
Ontario                                         Suicidal thoughts, 12 months   
Prince Edward Island          Other drug use (excluding cannabis), 12 months   
Quebec                                          Suicidal thoughts, 12 months   
Saskatchewan               Any selected disorder (mental or substance), e...   

                          z-score Actual rate  
Alberta                      1.06         7.1  
British Columbia       

In [55]:
rows = []
for ind in keep:
    sub = clean[clean['Indicators'] == ind].rename(
        columns={'Unemployment Rate': 'unemp', 'Mental Health Percent': 'mh', 'Age group': 'age'})
    m = smf.ols('mh ~ unemp * C(age) + C(Gender) + C(Year)', data=sub).fit()
    print(f"\n{ind}")
    print(m.summary().tables[1])


Cannabis use, 12 months
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                            30.6065      2.887     10.603      0.000      24.891      36.322
C(age)[T.25 to 44 years]            -10.9090      3.126     -3.490      0.001     -17.098      -4.720
C(age)[T.45 to 64 years]            -21.0919      3.379     -6.242      0.000     -27.782     -14.402
C(age)[T.65 years and over]         -28.5742     12.512     -2.284      0.024     -53.347      -3.802
C(Gender)[T.Women+]                  -7.1006      0.981     -7.240      0.000      -9.043      -5.159
C(Year)[T.2012]                       0.0823      1.027      0.080      0.936      -1.951       2.116
C(Year)[T.2022]                      11.2558      1.471      7.654      0.000       8.344      14.167
unemp                                 0.1123      0.181  

In [56]:
for ind in keep:
    sub = clean[clean['Indicators'] == ind].rename(
        columns={'Unemployment Rate': 'unemp', 'Mental Health Percent': 'mh', 'Age group': 'age'})
    m = smf.ols('mh ~ unemp * C(Year) + C(age) + C(Gender)', data=sub).fit()
    print(f"\n{ind}")
    print(m.summary().tables[1])


Cannabis use, 12 months
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept                      30.9452      2.418     12.797      0.000      26.158      35.732
C(Year)[T.2012]                 4.1047      2.055      1.997      0.048       0.036       8.174
C(Year)[T.2022]                23.5281      2.968      7.926      0.000      17.651      29.405
C(age)[T.25 to 44 years]      -14.4957      1.394    -10.399      0.000     -17.255     -11.736
C(age)[T.45 to 64 years]      -23.5157      1.545    -15.225      0.000     -26.574     -20.458
C(age)[T.65 years and over]   -32.0304      2.717    -11.791      0.000     -37.409     -26.652
C(Gender)[T.Women+]            -7.4653      0.899     -8.307      0.000      -9.245      -5.686
unemp                           0.2130      0.157      1.354      0.178      -0.098       0.525
unemp:C(Year)[T

In [57]:
sub = clean[(clean['Indicators'] == 'Major depressive episode, 12 months') &
            (clean['Age group'] == '15 to 24 years')].rename(
    columns={'Unemployment Rate': 'unemp', 'Mental Health Percent': 'mh'})

m = smf.ols('mh ~ unemp * C(Year) + C(Gender)', data=sub).fit()
print(sub.shape)
print(m.summary().tables[1])

(24, 7)
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 2.6599      5.090      0.523      0.608      -8.080      13.400
C(Year)[T.2012]           7.3678      5.696      1.294      0.213      -4.649      19.385
C(Year)[T.2022]           7.8962      6.788      1.163      0.261      -6.425      22.217
C(Gender)[T.Women+]       4.6185      1.410      3.277      0.004       1.645       7.592
unemp                     0.0803      0.352      0.228      0.823      -0.663       0.824
unemp:C(Year)[T.2012]    -0.4453      0.431     -1.034      0.316      -1.354       0.463
unemp:C(Year)[T.2022]     0.0298      0.616      0.048      0.962      -1.270       1.329


In [58]:
# Core findings
robust.to_csv('robustness_table.csv', index=False)  # already saved

# Age story
by_age.to_csv('by_age.csv')
by_age_unemp.to_csv('by_age_unemp.csv')

# Year reversal — the key new finding
year_results = []
for ind in keep:
    sub = clean[clean['Indicators'] == ind].rename(
        columns={'Unemployment Rate': 'unemp', 'Mental Health Percent': 'mh', 'Age group': 'age'})
    m = smf.ols('mh ~ unemp * C(Year) + C(age) + C(Gender)', data=sub).fit()
    for yr, term in [('2002', 'unemp'), ('2012', 'unemp:C(Year)[T.2012]'), ('2022', 'unemp:C(Year)[T.2022]')]:
        if term in m.params:
            slope = m.params['unemp'] + (m.params[term] if term != 'unemp' else 0)
            year_results.append({'Indicator': ind, 'Year': yr, 'Slope': slope, 'P': m.pvalues[term]})
pd.DataFrame(year_results).to_csv('year_slopes.csv', index=False)

# Age interaction — the youth-depression finding
age_results = []
for ind in keep:
    sub = clean[clean['Indicators'] == ind].rename(
        columns={'Unemployment Rate': 'unemp', 'Mental Health Percent': 'mh', 'Age group': 'age'})
    m = smf.ols('mh ~ unemp * C(age) + C(Gender) + C(Year)', data=sub).fit()
    for grp in ['15 to 24 years', '25 to 44 years', '45 to 64 years', '65 years and over']:
        term = f'unemp:C(age)[T.{grp}]'
        slope = m.params['unemp'] + (m.params[term] if term in m.params else 0)
        age_results.append({'Indicator': ind, 'Age group': grp, 'Slope': slope})
pd.DataFrame(age_results).to_csv('age_slopes.csv', index=False)

# Province scale and signature
avg_rank.to_csv('province_rank.csv')
signature_df.to_csv('province_signature.csv')